# Calculate auROC for first vs not-first lapses

Kendra Wyant  
May 17, 2026

### Set Up Environment

Packages, functions, and paths

In [ ]:

library(tidyverse)


── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.2.0     ✔ readr     2.2.0
✔ forcats   1.0.1     ✔ stringr   1.6.0
✔ ggplot2   4.0.2     ✔ tibble    3.3.1
✔ lubridate 1.9.5     ✔ tidyr     1.3.2
✔ purrr     1.2.1     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors

── Attaching packages ────────────────────────────────────── tidymodels 1.4.1 ──
✔ broom        1.0.12     ✔ rsample      1.3.2 
✔ dials        1.4.2      ✔ tailor       0.1.0 
✔ infer        1.1.0      ✔ tune         2.0.1 
✔ modeldata    1.5.1      ✔ workflows    1.3.0 
✔ parsnip      1.4.1      ✔ workflowsets 1.1.1 
✔ recipes      1.3.1      ✔ yardstick    1.3.2 
── Conflicts ───────────────────────────────────────── tidymodels_conflicts() ──
✖ scales::discard() masks purrr::discard()
✖ dplyr::filter()   masks stats::filter()
✖ recipes::fixed()  masks stringr::fixed()
✖ dplyr::lag()      masks stats::lag()
✖ yardstick::spec() masks readr::spec()
✖ recipes::step()   masks stats::step()

ℹ SHA-1 hash of file is "0faa14c0c44c2635216370888b7da9bfa8d07979"

### Data

In [ ]:
preds <- read_rds(here::here(path_models,
                             "outer_preds_6_x_5_1_x_5_day_v10_nested_full.rds"))

labels <- read_csv(here::here(path_shared, "lapse_labels_24h_day_gps.csv"),
                   show_col_types = FALSE)


### Create first lapse data frame

In [ ]:
first_lapses <- labels |> 
  filter(lapse == "Lapse") |> 
  group_by(subid) |> 
  arrange(window_start) |> 
  slice_head(n = 1)


### Combine data

In [ ]:
preds_combined <- labels |> 
  mutate(first_lapse = if_else(label_num %in% first_lapses$label_num, "Yes", "No")) |> 
  select(id_obs = label_num,
         label = lapse,
         subid,
         first_lapse) |>
  right_join(preds, by = c("id_obs", "label")) |> 
  mutate(label = factor(label, levels = c("Lapse", "No lapse")))

# nrow(first_lapses)
# preds_combined |> filter(first_lapse == "Yes") |> nrow() |>  unique()

# nrow(preds)
# nrow(preds_combined) # should be same as nrow(preds)


### Calculate auROCs comparing up first lapse to no lapse and not first lapse to no lapse

First lapses

In [ ]:
auroc_first_lapse <- preds_combined |> 
  filter(first_lapse == "Yes" | label == "No lapse") |> 
  nest(.by = outer_split_num, .key = "preds") |> 
  mutate(auroc = map(preds, \(preds) roc_auc(preds, prob_raw, 
                                             truth = label))) |> 
  select(-preds) |> 
  unnest(auroc) |> 
  select(-c(.estimator, .metric)) |> 
  rename(first_lapse = .estimate) 
     
auroc_not_first_lapse <- preds_combined |> 
  filter(first_lapse == "No") |> 
  nest(.by = outer_split_num, .key = "preds") |> 
  mutate(auroc = map(preds, \(preds) roc_auc(preds, prob_raw, 
                                             truth = label))) |> 
  select(-preds) |> 
  unnest(auroc) |> 
  select(-c(.estimator, .metric)) |> 
  rename(not_first_lapse = .estimate) 

aurocs <- auroc_first_lapse |> 
  left_join(auroc_not_first_lapse, by = "outer_split_num") |> 
  arrange(outer_split_num) |> 
  glimpse()


Rows: 30
Columns: 3
$ outer_split_num <int> 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16,…
$ first_lapse     <dbl> 0.7942752, 0.8092201, 0.7645230, 0.7025159, 0.7932255,…
$ not_first_lapse <dbl> 0.9494416, 0.9607761, 0.9426066, 0.9580109, 0.9588243,…

In [ ]:
round(median(aurocs$first_lapse), 2)


[1] 0.75

[1] 0.96